In [ ]:
import os
import shutil
from tqdm import tqdm

In [ ]:
from avapi.carla import CarlaScenesManager


# cpath = os.path.join("/data/shared/CARLA/multi-agent-intersection/")
cpath = os.path.join("/data/shared/CARLA/multi-agent-trust/")
CSM = CarlaScenesManager(cpath)
idx = 0
CDM = CSM.get_scene_dataset_by_index(idx)
vid_folder = f"videos_intersection_{idx}"

print(CSM.scenes)
print(f"{len(CDM)} frames")

In [ ]:
import matplotlib.pyplot as plt
from avapi.visualize.snapshot import show_lidar_bev_with_boxes

# load point cloud
lidar_sensor = "lidar-0"
agent = 0
frame_idx = 10
frame = CDM.get_frames(sensor=lidar_sensor, agent=agent)[frame_idx]
pc = CDM.get_lidar(frame=frame, sensor=lidar_sensor, agent=agent)
show_lidar_bev_with_boxes(pc=pc)

# run concave hull algorithm
hull = pc.concave_hull_bev(concavity=2, length_threshold=5)
plt.plot(-1 * hull[:, 1], hull[:, 0])
plt.axis("scaled")
plt.show()

## Test Multi-Agent Tracking and Fusion

In [ ]:
from avstack.datastructs import DataContainer
from avstack.geometry import GlobalOrigin3D
from avstack.modules.perception.object3d import MMDetObjectDetector3D
from avstack.modules.perception.detections import BoxDetection
from avstack.modules.tracking.tracker3d import BasicBoxTracker3D
from avstack.modules.tracking.multisensor import MeasurementBasedMultiTracker


from mate import plotting
from mate.estimator import TrustEstimator
from mate.utils import BetaDistWriter, PsmWriter


# init models
# agents = list(range(len(CDM.get_agents(frame=1))))
agents = list(range(4))
agent_is_static = {
    i: "static" in CDM.get_agent(frame=1, agent=i).obj_type for i in agents
}
n_static = sum(list(agent_is_static.values()))
print(
    "There are {} agents\n   {} mobile, {} static".format(
        len(agents), len(agents) - n_static, n_static
    )
)
percep_veh = MMDetObjectDetector3D(model="pointpillars", dataset="carla-vehicle", gpu=0)
percep_inf = MMDetObjectDetector3D(
    model="pointpillars", dataset="carla-infrastructure", gpu=0
)
percep_col = None
trackers = {agent: BasicBoxTracker3D() for agent in agents}
trackers["central"] = MeasurementBasedMultiTracker(
    tracker=BasicBoxTracker3D(name="trackerformulti")
)
trackers["collab"] = BasicBoxTracker3D()

# init data structures
ss_tracks = {}
dets = {agent: [] for agent in agents}
tracks = {"central": [], **{agent: [] for agent in agents}}
timestamps_all = {agent: [] for agent in agents}
imgs_all = {agent: [] for agent in agents}
pcs_all = {agent: [] for agent in agents}
dets_all = {agent: [] for agent in agents}
dets_all["collab"] = []
tracks_all = {agent: [] for agent in agents}
tracks_all["central"] = []
tracks_all["collab"] = []
agent_0_frames = CDM.get_frames(sensor="lidar-0", agent=0)[1:-1]
platforms_all = {agent: [] for agent in agents}

# flags for this run
run_perception_model = True
run_distributed_perception = True
run_distributed_tracking = True
run_centralized_tracking = True
run_collaborative_perception = False
run_collaborative_tracking = False
run_trust_estimation = True
debug_trust = True
psm_agent_file = "psm_agent_log.txt"
open(psm_agent_file, "w").close()
psm_track_file = "psm_track_log.txt"
open(psm_track_file, "w").close()
agent_trust_file = "trust_agent_log.txt"
open(agent_trust_file, "w").close()
track_trust_file = "trust_track_log.txt"
open(track_trust_file, "w").close()
if os.path.exists("debug_figures"):
    shutil.rmtree("debug_figures")

if run_trust_estimation:
    trustest = TrustEstimator()
    trust_burnin = 5

# run loop
n_frames_max = 100
ego_agent = agents[0]
for i_frame, frame in enumerate(
    tqdm(agent_0_frames[: min(n_frames_max, len(agent_0_frames))])
):
    fovs = {}
    fovs_local = {}
    platforms = {}
    perception_input = {}
    for agent in agents:
        ###############################################
        # GET DATA
        ###############################################
        lidar_sensor = "lidar-0"
        camera_sensor = "camera-0"
        timestamp = CDM.get_timestamp(frame=frame, sensor=lidar_sensor, agent=agent)
        img = CDM.get_image(frame=frame, sensor=camera_sensor, agent=agent)
        pc = CDM.get_lidar(frame=frame, sensor=lidar_sensor, agent=agent)
        imgs_all[agent].append(img)
        pcs_all[agent].append(pc)
        objs = CDM.get_objects(frame=frame, sensor=lidar_sensor, agent=agent)
        calib = CDM.get_calibration(frame=frame, sensor=lidar_sensor, agent=agent)
        fovs_local[agent] = pc.concave_hull_bev(
            concavity=1, length_threshold=4, in_global=False
        )
        fovs[agent] = pc.concave_hull_bev(
            concavity=1, length_threshold=4, in_global=True
        )
        platforms[agent] = calib.reference
        platforms_all[agent].append(calib.reference)

        ###############################################
        # DISTRIBUTED PERCEPTION
        ###############################################
        if run_distributed_perception:
            if run_perception_model:
                if agent_is_static[agent]:
                    dets[agent] = percep_inf(pc)
                else:
                    dets[agent] = percep_veh(pc)
            else:  # use groundtruth
                dets[agent] = DataContainer(
                    frame=frame,
                    timestamp=timestamp,
                    source_identifier="perception",
                    data=[
                        BoxDetection(
                            source_identifier="perception",
                            box=obj.box3d,
                            reference=obj.reference,
                            obj_type=obj.obj_type,
                            score=1.0,
                        )
                        for obj in objs
                        # if obj.ID == 15
                    ],
                )
            dets_all[agent].append(dets[agent])

        ###############################################
        # DISTRIBUTED TRACKING USING DISTRIBUTED PERCEP
        ###############################################
        if run_distributed_tracking:
            assert run_distributed_perception
            tracks[agent] = trackers[agent](
                dets[agent], platform=calib.reference, calibration=calib
            )
            if not isinstance(tracks[agent], DataContainer):
                raise
            tracks_all[agent].append([track.box3d.copy() for track in tracks[agent]])

    ###############################################
    # COLLABORATIVE PERCEPTION/TRACKING
    ###############################################
    if run_collaborative_perception:
        raise NotImplementedError

    ###############################################
    # CENTRALIZED TRACKING USING DISTRIBUTED PERCEP
    ###############################################
    # run central tracker on all detections
    if run_centralized_tracking:
        # Run trust model
        agent_positions = {
            ID: platforms[ID].integrate(start_at=GlobalOrigin3D).x
            for ID, p in platforms.items()
        }
        obj_positions = [
            obj.position.change_reference(GlobalOrigin3D, inplace=False).x
            for obj in objs
        ]
        dets_global = {
            ID: dets_agent.apply_and_return(
                "change_reference", GlobalOrigin3D, inplace=False
            )
            for ID, dets_agent in dets.items()
        }
        tracks_global = {
            ID: (
                tracks_agent.apply_and_return(
                    "change_reference", GlobalOrigin3D, inplace=False
                )
                if len(tracks_agent) > 0
                else []
            )
            for ID, tracks_agent in tracks.items()
        }

        # Propagate central tracks to current time
        for track in tracks_global["central"]:
            track.predict(timestamp)

        # allow for tracker burnin
        if i_frame >= trust_burnin:
            clusters, psms_agents, psms_tracks = trustest(
                agent_positions,
                fovs,
                dets_global,
                tracks_global.get("central", []),
                tracks_global,
            )

            # debugging
            if debug_trust:
                PsmWriter.write(frame, psms_agents, psm_agent_file)
                PsmWriter.write(frame, psms_tracks, psm_track_file)
                BetaDistWriter.write(frame, trustest.agent_trust, agent_trust_file)
                BetaDistWriter.write(frame, trustest.track_trust, track_trust_file)
                plotting.plot_agents_detections(
                    agent_positions,
                    fovs,
                    dets_global,
                    show=False,
                    save=True,
                    fig_dir="debug_figures/detections",
                    suffix=f"-frame-{frame}",
                    extension="png",
                )
                plotting.plot_agents_tracks(
                    agent_positions,
                    fovs,
                    tracks["central"],
                    show=False,
                    save=True,
                    fig_dir="debug_figures/tracks",
                    suffix=f"-frame-{frame}",
                    extension="png",
                )
                plotting.plot_trust(
                    trustest,
                    show=False,
                    save=True,
                    fig_dir="debug_figures/trust",
                    use_subfolders=True,
                    suffix=f"-frame-{frame}",
                    extension="png",
                )
                plt.close()

            # if i_frame > 15:
            #     break

        # Run tracking
        tracks["central"] = trackers["central"](
            detections=dets_global,
            fovs=fovs,
            platforms=[GlobalOrigin3D] * len(fovs),
        )
        tracks_all["central"].append(
            [track.box3d.copy() for track in tracks["central"]]
        )

In [ ]:
from mate import plotting

# plotting.plot_agents(agent_positions, fovs)
# plotting.plot_agents_objects(agent_positions, fovs, obj_positions, fps=[], fns=[])
plotting.plot_agents_detections(agent_positions, fovs, dets_global)
plotting.plot_agents_clusters(agent_positions, fovs, clusters)
plotting.plot_agents_tracks(agent_positions, fovs, tracks["central"])

In [ ]:
from mate import plotting

plotting.plot_trust(trustest)

## Visualize

In [ ]:
from utils import make_agent_movies, make_central_movie, make_collab_movie

# agent-specific movie
if run_distributed_tracking:
    print("Making distributed agent movies")
    extent = [[-70, 70], [-70, 70], [-100, 100]]
    for agent in agents:
        # AVstack movie
        make_agent_movies(
            imgs=imgs_all[agent],
            pcs=pcs_all[agent],
            dets=dets_all[agent],
            tracks=tracks_all[agent],
            agent=agent,
            extent=extent,
            percep_movies=False,
            track_movies=True,
            img_movies=True,
            bev_movies=True,
            vid_folder=vid_folder,
        )


# central tracking movie
if run_centralized_tracking:
    print("Making centralized tracking movies")
    extent = [[-150, 20], [-80, 40], [-100, 100]]
    # Make central BEV movie
    make_central_movie(
        pcs_all,
        tracks_all["central"],
        ego=platforms_all[ego_agent],
        extent=extent,
        vid_folder=vid_folder,
        colormethod="channel-4",
    )
    # Make central marginal movies
    print("Making centralized tracking movie marginals")
    extent = [[-70, 70], [-70, 70], [-100, 100]]
    for agent in agents:
        # AVstack movie
        make_agent_movies(
            imgs=imgs_all[agent],
            pcs=pcs_all[agent],
            dets=[],
            tracks=tracks_all["central"],
            agent=agent,
            extent=extent,
            percep_movies=False,
            track_movies=True,
            img_movies=True,
            bev_movies=True,
            vid_folder=vid_folder,
            suffix="-from-central",
        )


# collaborative perception movie
if run_collaborative_tracking:
    print("Making collaborative tracking movies")
    extent = [[-150, 20], [-80, 40], [-100, 100]]
    make_collab_movie(
        pcs_all,
        tracks_all["collab"],
        ego=platforms_all[ego_agent],
        extent=extent,
        vid_folder=vid_folder,
        colormethod="channel-4",
    )